# 자세 판단을 위한 Threshold 정하기

## 데이터 불러오기 및 설정

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [ ]:
# 좋은 자세 데이터셋 하나로 합치기
data_good = []
for i in range(1, 5):
  data = np.load(f"/content/dataset_good_{i}.npz")
  data_good.append(data)

In [ ]:
data_good

[NpzFile '/content/dataset_good_1.npz' with keys: sequences, labels,
 NpzFile '/content/dataset_good_2.npz' with keys: sequences, labels,
 NpzFile '/content/dataset_good_3.npz' with keys: sequences, labels,
 NpzFile '/content/dataset_good_4.npz' with keys: sequences, labels]

In [ ]:
# 나쁜 자세 데이터셋 불러오기
data_bad = np.load("/content/dataset_bad_1.npz")

In [ ]:
data_bad

NpzFile '/content/dataset_bad_1.npz' with keys: sequences, labels

`num_features` - feature 개수를 나타내는 변수

`feature_context`  - feature의 내용을 나타내는 딕셔너리

In [ ]:
# feature 개수
num_features = 6

In [ ]:
feature_context = {
    0: '왼손목-왼어깨 거리',
    1: '양쪽 팔꿈치 거리',
    2: '왼손목 속도',
    3: '왼팔 각도',
    4: '오른팔 각도',
    5: '오른손목 각도'
}

In [ ]:
feature_context[0]

'왼손목-왼어깨 거리'

`all_seq_good` - 전체 좋은 자세 데이터셋

`seq_good`  - 1개의 좋은 자세 데이터셋

In [ ]:
all_seq_good = np.concatenate(
    [data_good[i]['sequences'] for i in range(4)],
    axis=0
)

all_seq_good.shape # (548*4, 30, 6)

(2743, 30, 6)

In [ ]:
seq_good = data_good[0]['sequences']
seq_good

array([[[2.0859554e-01, 3.5257137e-01, 2.6717852e-03, 4.2829861e+01,
         2.9796011e+01, 1.6600162e+02],
        [2.0727530e-01, 3.5843712e-01, 6.3225912e-04, 4.2032024e+01,
         2.9986713e+01, 1.6701152e+02],
        [2.0685612e-01, 3.7626326e-01, 2.4106103e-04, 4.1958458e+01,
         7.5007149e+01, 1.4582460e+02],
        ...,
        [1.9098088e-01, 3.6362013e-01, 1.5909479e-04, 3.8577003e+01,
         4.3485294e+01, 1.7550623e+02],
        [1.9118676e-01, 3.6546034e-01, 8.5371918e-05, 3.8551006e+01,
         6.1369057e+01, 1.3735817e+02],
        [1.9275945e-01, 3.4988800e-01, 3.2315220e-04, 3.8991386e+01,
         3.9470348e+01, 1.7984354e+02]],

       [[2.0859554e-01, 3.5257137e-01, 2.6717852e-03, 4.2829861e+01,
         2.9796011e+01, 1.6600162e+02],
        [2.0727530e-01, 3.5843712e-01, 6.3225912e-04, 4.2032024e+01,
         2.9986713e+01, 1.6701152e+02],
        [2.0685612e-01, 3.7626326e-01, 2.4106103e-04, 4.1958458e+01,
         7.5007149e+01, 1.4582460e+02],
    

## Feature별 분석 및 threshold 설정

- **거리 기반 feature** (왼손목-왼어깨, 양쪽 팔꿈치)

연주자의 체격에 따라 달라질 수 있어 신체 비율로 정규화 고려.
`왼손목-왼어깨` 포지션 이동 시를 제외하고 일정 범위를 유지해야 한다. `양쪽 팔꿈치` 보잉 시 이 거리가 급격히 좁아지거나 넓어지면 자세가 무너진 것(ex. 정상 범위의 15% 이탈 시 경고)으로 판단한다.

- **각도 기반 feature** (왼팔, 오른팔, 오른손목)

`오른팔/오른손목` 활의 위치에 따라 이상적인 각도 경로가 존재한다.

정적: 오른팔 각도가 160도를 넘어가면 팔을 과하게 폈다는 식의 hard threshold를 설정할 수 있다.

- **속도 기반 feature** (왼손목 속도)

포지션 이동이 없는 구간에서 속도가 일정 임계값을 넘으면 불필요한 흔들림으로 간주한다.

### 자세 점수 로직

**6개 feature를 조합한 '자세 점수' 로직**

"왼손목 거리 차이"와 "왼팔 각도"를 결합하면 훨씬 고도화된 피드백이 가능.

- Case A (안정적): 왼팔 각도가 정상 범위 내에 있고, 왼손목 거리 차이가 작음 → "완벽합니다!"
- Case B (불안정): 왼팔 각도는 정상인데, 왼손목 거리 차이가 큼 → "팔꿈치는 고정되었으나 손목이 흔들립니다."
- Case C (자세 무너짐): 왼팔 각도가 범위를 벗어나면서 손목 거리 차이도 커짐 → "전체적인 자세가 무너졌습니다. 어깨와 팔의 힘을 빼세요."